In [1]:
import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from web_content_scraper import fetch_website_links, fetch_website_contents
from openai import OpenAI

In [ ]:
load_dotenv(override=True)

base_url = "https://generativelanguage.googleapis.com/v1beta/openai/"
api_key = os.getenv('GEMINI_API_KEY')

MODEL = 'gemini-3.6-flash'

gemini = OpenAI(
    base_url = base_url, 
    api_key = api_key
)

In [ ]:
link_system_prompt = """
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relevant to include in a brochure about the company,
such as links to an About page, or a Company page, or Careers/Jobs pages.
You should respond in JSON as in this example:

{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
    ]
}
"""

In [ ]:
def get_links_user_prompt(url):
    user_prompt = f"""
Here is the list of links on the website {url} -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

"""
    links = fetch_website_links(url)
    user_prompt += "\n".join(links)
    return user_prompt

In [ ]:
def select_relevant_links(url):
    
    response = gemini.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    
    return links

In [ ]:
def fetch_page_and_all_relevant_links(url):
    contents = fetch_website_contents(url)
    relevant_links = select_relevant_links(url)
    result = f"## Landing Page:\n\n{contents}\n## Relevant Links:\n"
    for link in relevant_links['links']:
        result += f"\n\n### Link: {link['type']}\n"
        result += fetch_website_contents(link["url"])
    return result

In [ ]:
brochure_system_prompt = """
You are an assistant that analyzes the contents of several relevant pages from a company website
and creates a short brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information.
"""



In [ ]:
def get_brochure_user_prompt(company_name, url):
    user_prompt = f"""
You are looking at a company called: {company_name}
Here are the contents of its landing page and other relevant pages;
use this information to build a short brochure of the company in markdown without code blocks.\n\n
"""
    user_prompt += fetch_page_and_all_relevant_links(url)
    user_prompt = user_prompt[:5_000] # Truncate if more than 5,000 characters
    return user_prompt

In [ ]:
def create_brochure(company_name, url):
    print(f"Creating Brochure for  {company_name} by calling {MODEL}")
    response = gemini.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
        ],
    )
    result = response.choices[0].message.content
    
    display(Markdown(result))
    return result

In [118]:
html_system_prompt = """
You are an expert document designer for PDF conversion.
Convert the provided Markdown brochure into a clean, standalone HTML document for xhtml2pdf.

CRITICAL PDF ENGINE RULES:
1. DO NOT use <table> tags. Represent lists and structured data using <ul>, <li>, <div>, and heading tags (<h1>, <h2>, <h3>).
2. DO NOT use CSS variables (like var(...)). Use direct HEX color codes (e.g., #1e40af, #1e293b, #f8fafc, #333333).
3. DO NOT use @font-face or external font imports.
4. Include @page { size: letter; margin: 0.75in; } in the <style> block.
5. Return ONLY the raw HTML string inside <html>...</html> without markdown backticks.
"""



In [119]:
from xhtml2pdf import pisa

def convert_brochure_to_pdf(brochure_markdown, company_name, pdf_filename="brochure.pdf"):
    print("Converting brochure content to styled HTML...")
    
    html_response = gemini.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": html_system_prompt},
            {"role": "user", "content": f"Company: {company_name}\n\nBrochure:\n{brochure_markdown}"}
        ],
    )
    
    html_content = html_response.choices[0].message.content
    
    # Clean any ```html backticks from LLM output
    if "```" in html_content:
        html_content = html_content.split("```html")[-1].split("```")[0].strip()
        
    print(f"Saving PDF to {pdf_filename}...")
    with open(pdf_filename, "wb") as f:
        pisa_status = pisa.CreatePDF(html_content, dest=f)
        
    if pisa_status.err:
        print("Warning: PDF rendered with minor layout warnings.")
    else:
        print(f"Successfully created PDF: {pdf_filename}")


In [122]:
# 1. Get the brochure content
brochure_text = create_brochure("Smile Dental Clinics", "https://www.smiledentalclinics.com")

# 2. Pass the clean brochure output into the PDF converter
convert_brochure_to_pdf(brochure_text, "Smile Dental Clinics", "Smile_Dental_Clinics.pdf")


Creating Brochure for  Smile Dental Clinics by calling gemini-3.6-flash


# Smile Dental Clinics
**Correcting Smiles Since 2000**

---

## Welcome to Smile Dental Clinics

For over two decades, **Smile Dental Clinics** has been a trusted leader in gentle, affordable, and high-quality oral healthcare across the Phoenix Metropolitan Area. Led by Dr. Eddie Harsini, DDS, our practice combines advanced dental technology with a warm, compassionate approach to deliver exceptional care to every patient.

Whether you are looking for routine preventative care, cosmetic transformations, or complex restorative procedures, Smile Dental Clinics is dedicated to keeping your smile healthy and bright.

---

## About Our Practice & Growth

* **Established:** 2000
* **Leadership:** Eddie Harsini, DDS
* **Community Impact:** Over **45,000 patients** served Valley-wide
* **Recognition:** Awarded *Best of Business Rate 2025*
* **Service Areas:** Phoenix, Goodyear, Alhambra, Peoria, Tolleson, and Glendale, AZ

Built on decades of clinical excellence, modern infrastructure, and continuous professional training, Smile Dental Clinics offers patients high-end dental procedures at a reasonable cost.

---

## Our Culture & Patient Experience

At Smile Dental Clinics, we believe that visiting the dentist should be a positive, stress-free experience. Our team prioritizes:

* **Patient-Centered Care:** We specialize in addressing dental fear and anxiety through gentle techniques and sedation options.
* **Warm & Welcoming Atmosphere:** From our front desk staff to our experienced dentists, our team is known for being friendly, approachable, and caring.
* **Accessibility & Convenience:** We offer online payments, flexible financing options, video patient education, and emergency dental services.

> *"Amazing Experience! Not only was the staff nice, warm and welcoming with smiles, but the doctors are so gentle, knowledgeable and caring. I recommend this great place anytime."*  
> — **Behnoosh, Happy Patient**

---

## Comprehensive Dental Treatments

We offer a full spectrum of dental care under one roof using state-of-the-art technology:

* **General & Preventative Care:** Exams, X-rays, cleanings, fillings, sealants, root canals, and tooth extractions.
* **Cosmetic Dentistry:** Invisalign®, teeth whitening, and custom veneers.
* **Restorative & Advanced Care:** Dental crowns, bridges, partial/full dentures, dental implants, and All-on-4 / All-on-X procedures.
* **Specialized Care:** Emergency dental services, sedation dentistry, and treatments for teeth grinding.

---

## Careers at Smile Dental Clinics

As a growing practice with deep roots in the community, Smile Dental Clinics is always seeking dedicated, compassionate professionals to join our team. 

### Why Join Us?
* **Collaborative Environment:** Work alongside experienced clinicians and supportive team members who value patient well-being above all else.
* **Modern Facilities:** Gain experience with cutting-edge dental technologies and modern practices.
* **Professional Development:** Benefit from ongoing training and career growth opportunities within a highly respected practice.

If you are passionate about helping patients smile with confidence, explore our career opportunities today!

---

## Contact & Appointments

Ready to experience the Smile Dental Clinics difference? Schedule a consultation or reach out to our team:

* **Phone:** (623) 257-7475
* **Locations:** Serving Phoenix and surrounding Valley communities
* **Appointments:** Request an appointment online or call today!

Converting brochure content to styled HTML...
Saving PDF to Smile_Dental_Clinics.pdf...
Successfully created PDF: Smile_Dental_Clinics.pdf
